In [1]:
import pandas as pd
from datasets import load_dataset

pd.set_option("display.max_colwidth", 300)

ds = load_dataset("pyupeu/social-media-peruvian-sentiment")

df_all = pd.concat(
    [ds[split].to_pandas().assign(split_original=split) for split in ds.keys()],
    ignore_index=True
)

print("Total de registros en el pool unificado:", len(df_all))
print(df_all["split_original"].value_counts())
df_all.head()

README.md:   0%|          | 0.00/857 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  990kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  306kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Generating train split:   0%|          | 0/9336 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2918 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2335 [00:00<?, ? examples/s]

Total de registros en el pool unificado: 14589
split_original
train         9336
validation    2918
test          2335
Name: count, dtype: int64


,text,label,label_name,split_original
0,"Salió para Aser farándula ese chato, solo quiere llamar la atención 😂 q trabaje bago , aragán",0,negative,train
1,Si así eres tu también!!! ... Cuando llegaste a Cancun al hotel donde trabajaba.. Y la netta casi todos son bien mamones.. Y otros mamones y piojos jajaj pero bueno en sus videos son otra cosa👎🏻,0,negative,train
2,Nunca se tiene pocos zapatos... 🤣 🤣 🤣 🤣 Tengo algunos tacones y botas que aun no los e usado y sigo comprando mas.,1,neutral,train
3,Queen Reigne te 😂😂😂 sorry na,1,neutral,train
4,En media hora comienza el toque de queda dijo,1,neutral,train


## Eliminar textos con etiquetas contraadictorias

In [2]:
# Identificar textos duplicados (en cualquier split) con más de una etiqueta distinta
revision_duplicados = (
    df_all[df_all["text"].duplicated(keep=False)]
    .groupby("text")
    .agg(
        cantidad=("text", "count"),
        etiquetas=("label_name", lambda x: sorted(set(x))),
        splits=("split_original", lambda x: sorted(set(x)))
    )
    .reset_index()
)

revision_duplicados["num_etiquetas"] = revision_duplicados["etiquetas"].apply(len)

textos_contradictorios = revision_duplicados[
    revision_duplicados["num_etiquetas"] > 1
]["text"].tolist()

print("Textos con etiquetas contradictorias encontrados:", len(textos_contradictorios))

df_sin_contradicciones = df_all[~df_all["text"].isin(textos_contradictorios)].copy()

print("Antes:", len(df_all))
print("Después de eliminar contradictorios:", len(df_sin_contradicciones))

Textos con etiquetas contradictorias encontrados: 3
Antes: 14589
Después de eliminar contradictorios: 14583


## Eliminar duplicados simples, nos quedamos con una sola ocurrencia por texto único

In [3]:
duplicados_restantes_antes = df_sin_contradicciones["text"].duplicated().sum()
print("Duplicados restantes antes de deduplicar:", duplicados_restantes_antes)

df_clean = (
    df_sin_contradicciones
    .drop_duplicates(subset=["text"], keep="first")
    .reset_index(drop=True)
)

print("Antes de deduplicar:", len(df_sin_contradicciones))
print("Después de deduplicar:", len(df_clean))
print("Duplicados restantes:", df_clean["text"].duplicated().sum())

Duplicados restantes antes de deduplicar: 39
Antes de deduplicar: 14583
Después de deduplicar: 14544
Duplicados restantes: 0


## Limpiar URLs y el símbolo # de hashtags (se conservan menciones). Además las tildes se mantienen

In [5]:
import re

def limpiar_urls_hashtags_conservando_menciones(text):
    if not isinstance(text, str):
        return ""

    # 1. Eliminar URLs reales
    text = re.sub(r"http\S+|www\.\S+|https?://\S+", " ", text, flags=re.IGNORECASE)

    # 2. Hashtags: quitar el símbolo # pero conservar la palabra
    text = re.sub(r"#(\w+)", r"\1", text)

    # 3. Normalizar espacios
    text = re.sub(r"\s+", " ", text).strip()

    return text

df_clean["text_clean"] = df_clean["text"].apply(limpiar_urls_hashtags_conservando_menciones)

# Verificamos que ya no queden URLs ni hashtags con #
patron_url_real = r"(?:http\S+|www\.\S+|https?://\S+)"
patron_hashtag = r"#\w+"
patron_mencion = r"@\w+"

print("URLs reales restantes:", df_clean["text_clean"].str.contains(patron_url_real, regex=True, case=False, na=False).sum())
print("Hashtags con # restantes:", df_clean["text_clean"].str.contains(patron_hashtag, regex=True, na=False).sum())
print("Menciones restantes:", df_clean["text_clean"].str.contains(patron_mencion, regex=True, na=False).sum())

URLs reales restantes: 0
Hashtags con # restantes: 0
Menciones restantes: 56


In [6]:
import re

# Ver una muestra de los textos con mención, tal cual quedaron
con_mencion = df_clean[df_clean["text_clean"].str.contains(r"@\w+", regex=True, na=False)]
print("Total de registros con mención:", len(con_mencion))

con_mencion[["text", "text_clean", "label_name"]].head(20)

Total de registros con mención: 56


,text,text_clean,label_name
28,"Chino dile a ese pelado que se deje de estarme pidiendo CANJE , como tiene para chupar como pendej@@o🥴🥴🥴 y no tiene para pagar su cuarto","Chino dile a ese pelado que se deje de estarme pidiendo CANJE , como tiene para chupar como pendej@@o🥴🥴🥴 y no tiene para pagar su cuarto",negative
787,Parece que la Shakira se puso doble to@ll@ higiénic@... Ya vuelta... 😎😎😎,Parece que la Shakira se puso doble to@ll@ higiénic@... Ya vuelta... 😎😎😎,neutral
1029,"¶Hace 45 años, nuestro 🇵🇪#PERÚ🇵🇪 desde Morales Bermúdes, pasando por Belaunde Terry, seguido por García Pérez; luego el hampón y genocida Japonés el Yakuza Kenya Inami Inamoto alias ""Alberto Fujimori"" con su siamés y secuas el criminal, delincuente y traídor arequipeño Ex agente de la CIA Judí...","¶Hace 45 años, nuestro 🇵🇪PERÚ🇵🇪 desde Morales Bermúdes, pasando por Belaunde Terry, seguido por García Pérez; luego el hampón y genocida Japonés el Yakuza Kenya Inami Inamoto alias ""Alberto Fujimori"" con su siamés y secuas el criminal, delincuente y traídor arequipeño Ex agente de la CIA JudíoEs...",negative
1454,"No cambias cabrilla, más palo eres chtm, !Yo me rectifió cuando me dé la gana! siiiiiii, toda una macinon@ tira mi3rd4, se te encogieron las boloñ@s 😬😬😂😂😂 aprovecha mientras tengas vitrina trde o temprano se te acabará o por viej@ ó porque ya no vendes con tus dices seguidores que tambien se hac...","No cambias cabrilla, más palo eres chtm, !Yo me rectifió cuando me dé la gana! siiiiiii, toda una macinon@ tira mi3rd4, se te encogieron las boloñ@s 😬😬😂😂😂 aprovecha mientras tengas vitrina trde o temprano se te acabará o por viej@ ó porque ya no vendes con tus dices seguidores que tambien se hac...",negative
1703,Buena la ensalada y felicidades para l@s héroes urban@s👏👏👏👏,Buena la ensalada y felicidades para l@s héroes urban@s👏👏👏👏,positive
2031,"Rojos izquierdistas y la Derecha corrupt@s la cras, pode dumbres de la politica peruana, venden humo, zangan@s, mermeler@s vividores de la teta del estado son la corruptela vendid@s al mejor postor junto a su mami chica k, fujiapristas, vitochos, etc, etc corrupt@s siguen vendiendo humo se creen...","Rojos izquierdistas y la Derecha corrupt@s la cras, pode dumbres de la politica peruana, venden humo, zangan@s, mermeler@s vividores de la teta del estado son la corruptela vendid@s al mejor postor junto a su mami chica k, fujiapristas, vitochos, etc, etc corrupt@s siguen vendiendo humo se creen...",negative
3237,"No tiene ni para tr@g@r, encima propicia la compra de celulares Hurtados, la firme chino le haces un mal a ese gordo llevandolo contigo! 🤦🏻‍♂️","No tiene ni para tr@g@r, encima propicia la compra de celulares Hurtados, la firme chino le haces un mal a ese gordo llevandolo contigo! 🤦🏻‍♂️",negative
3376,"Videaso, que tales monstruos!!! Todos a las @wsl carajo!!! 🙌 respect total","Videaso, que tales monstruos!!! Todos a las @wsl carajo!!! 🙌 respect total",negative
3456,"Luciano, tienes que pasar por el @Palacio del Sancochado Av. 28 de julio 990 - Cercado. También el Rinconcito Ferreñafano (frente al Congreso). Por el arroz con pato con ceviche de entrada 🙌🏻","Luciano, tienes que pasar por el @Palacio del Sancochado Av. 28 de julio 990 - Cercado. También el Rinconcito Ferreñafano (frente al Congreso). Por el arroz con pato con ceviche de entrada 🙌🏻",neutral
3593,"Veo muchas madres identificadas, pero nadie piensa en los hermanos mayores. Nosotr@s también somos parte del calvario 😂✨","Veo muchas madres identificadas, pero nadie piensa en los hermanos mayores. Nosotr@s también somos parte del calvario 😂✨",negative


In [7]:
# Separar cuáles son el placeholder "@username" y cuáles son otra cosa (posibles nombres reales)
es_placeholder = con_mencion["text_clean"].str.contains(r"@username\b", regex=True, na=False, case=False)

print("Con placeholder '@username':", es_placeholder.sum())
print("Con otro tipo de mención (revisar):", (~es_placeholder).sum())

con_mencion[~es_placeholder][["text", "text_clean", "label_name"]]

Con placeholder '@username': 0
Con otro tipo de mención (revisar): 56


,text,text_clean,label_name
28,"Chino dile a ese pelado que se deje de estarme pidiendo CANJE , como tiene para chupar como pendej@@o🥴🥴🥴 y no tiene para pagar su cuarto","Chino dile a ese pelado que se deje de estarme pidiendo CANJE , como tiene para chupar como pendej@@o🥴🥴🥴 y no tiene para pagar su cuarto",negative
787,Parece que la Shakira se puso doble to@ll@ higiénic@... Ya vuelta... 😎😎😎,Parece que la Shakira se puso doble to@ll@ higiénic@... Ya vuelta... 😎😎😎,neutral
1029,"¶Hace 45 años, nuestro 🇵🇪#PERÚ🇵🇪 desde Morales Bermúdes, pasando por Belaunde Terry, seguido por García Pérez; luego el hampón y genocida Japonés el Yakuza Kenya Inami Inamoto alias ""Alberto Fujimori"" con su siamés y secuas el criminal, delincuente y traídor arequipeño Ex agente de la CIA Judí...","¶Hace 45 años, nuestro 🇵🇪PERÚ🇵🇪 desde Morales Bermúdes, pasando por Belaunde Terry, seguido por García Pérez; luego el hampón y genocida Japonés el Yakuza Kenya Inami Inamoto alias ""Alberto Fujimori"" con su siamés y secuas el criminal, delincuente y traídor arequipeño Ex agente de la CIA JudíoEs...",negative
1454,"No cambias cabrilla, más palo eres chtm, !Yo me rectifió cuando me dé la gana! siiiiiii, toda una macinon@ tira mi3rd4, se te encogieron las boloñ@s 😬😬😂😂😂 aprovecha mientras tengas vitrina trde o temprano se te acabará o por viej@ ó porque ya no vendes con tus dices seguidores que tambien se hac...","No cambias cabrilla, más palo eres chtm, !Yo me rectifió cuando me dé la gana! siiiiiii, toda una macinon@ tira mi3rd4, se te encogieron las boloñ@s 😬😬😂😂😂 aprovecha mientras tengas vitrina trde o temprano se te acabará o por viej@ ó porque ya no vendes con tus dices seguidores que tambien se hac...",negative
1703,Buena la ensalada y felicidades para l@s héroes urban@s👏👏👏👏,Buena la ensalada y felicidades para l@s héroes urban@s👏👏👏👏,positive
2031,"Rojos izquierdistas y la Derecha corrupt@s la cras, pode dumbres de la politica peruana, venden humo, zangan@s, mermeler@s vividores de la teta del estado son la corruptela vendid@s al mejor postor junto a su mami chica k, fujiapristas, vitochos, etc, etc corrupt@s siguen vendiendo humo se creen...","Rojos izquierdistas y la Derecha corrupt@s la cras, pode dumbres de la politica peruana, venden humo, zangan@s, mermeler@s vividores de la teta del estado son la corruptela vendid@s al mejor postor junto a su mami chica k, fujiapristas, vitochos, etc, etc corrupt@s siguen vendiendo humo se creen...",negative
3237,"No tiene ni para tr@g@r, encima propicia la compra de celulares Hurtados, la firme chino le haces un mal a ese gordo llevandolo contigo! 🤦🏻‍♂️","No tiene ni para tr@g@r, encima propicia la compra de celulares Hurtados, la firme chino le haces un mal a ese gordo llevandolo contigo! 🤦🏻‍♂️",negative
3376,"Videaso, que tales monstruos!!! Todos a las @wsl carajo!!! 🙌 respect total","Videaso, que tales monstruos!!! Todos a las @wsl carajo!!! 🙌 respect total",negative
3456,"Luciano, tienes que pasar por el @Palacio del Sancochado Av. 28 de julio 990 - Cercado. También el Rinconcito Ferreñafano (frente al Congreso). Por el arroz con pato con ceviche de entrada 🙌🏻","Luciano, tienes que pasar por el @Palacio del Sancochado Av. 28 de julio 990 - Cercado. También el Rinconcito Ferreñafano (frente al Congreso). Por el arroz con pato con ceviche de entrada 🙌🏻",neutral
3593,"Veo muchas madres identificadas, pero nadie piensa en los hermanos mayores. Nosotr@s también somos parte del calvario 😂✨","Veo muchas madres identificadas, pero nadie piensa en los hermanos mayores. Nosotr@s también somos parte del calvario 😂✨",negative


In [8]:
import re

patron_mencion_real = r"(?<!\w)@\w+"   # @ NO precedido por una letra/número/guion_bajo

con_mencion["es_mencion_real"] = con_mencion["text_clean"].str.contains(patron_mencion_real, regex=True, na=False)

print("Menciones reales (usuario/persona):", con_mencion["es_mencion_real"].sum())
print("Uso de @ como letra dentro de palabra (censura/lenguaje inclusivo):", (~con_mencion["es_mencion_real"]).sum())

con_mencion[con_mencion["es_mencion_real"]][["text_clean", "label_name"]]

Menciones reales (usuario/persona): 24
Uso de @ como letra dentro de palabra (censura/lenguaje inclusivo): 32


/tmp/ipykernel_2470/560871147.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  con_mencion["es_mencion_real"] = con_mencion["text_clean"].str.contains(patron_mencion_real, regex=True, na=False)


,text_clean,label_name
28,"Chino dile a ese pelado que se deje de estarme pidiendo CANJE , como tiene para chupar como pendej@@o🥴🥴🥴 y no tiene para pagar su cuarto",negative
1029,"¶Hace 45 años, nuestro 🇵🇪PERÚ🇵🇪 desde Morales Bermúdes, pasando por Belaunde Terry, seguido por García Pérez; luego el hampón y genocida Japonés el Yakuza Kenya Inami Inamoto alias ""Alberto Fujimori"" con su siamés y secuas el criminal, delincuente y traídor arequipeño Ex agente de la CIA JudíoEs...",negative
3376,"Videaso, que tales monstruos!!! Todos a las @wsl carajo!!! 🙌 respect total",negative
3456,"Luciano, tienes que pasar por el @Palacio del Sancochado Av. 28 de julio 990 - Cercado. También el Rinconcito Ferreñafano (frente al Congreso). Por el arroz con pato con ceviche de entrada 🙌🏻",neutral
3687,❤️Vamos. Michi @micheilleSoifer si se puede estoy segura q vas a seguir en competencia yo te lo aseguro un saludo muy grande desde la. Ciudad de Casma,positive
4260,Quieren hacer Fraude Televisa y La Casa de los Famosos México Están protegiendo a su consentido !! Esto ya es Viral Fuera Poul) Fueraaaa!! 😤😤😤😤😤😤 Sergio Mayer Breton @ Emilio !!! Que Mexico alce la Voz💢💢💢💢💢💢@Niurka,negative
4731,"Saludos chicas , sería excelente ganar ese parlante de la gran industria de sonido Marshall debe sonar fabuloso ojalá tenga la suerte con @joaboloña la rubia hermosa...cuídense besos! MASMUSICACONMARSHALL in one 🌎",positive
4957,"Muy bien, @Gerardo Pe' da gusto cuando lo pones punche sigue así estuvo entretenido tienes un like 👍",positive
5510,@Un tombo pasó por ahí en una moto deportiva 🤣,neutral
5670,"Lo de ese asqueroso de Richard swing debe sancionarse si o si, pero no sean pendejos, en otros lados siempre hacían lo mismo y nadie decía nada...pero, Deplorable lo de Swing, pero vomitivo que lo condene carla garcia... la hija del ex-Presidente que contrató (de pura 🔥calentura 🔥) a una reconoc...",negative


In [9]:
def anonimizar_menciones_reales(text):
    return re.sub(r"(?<!\w)@\w+", "@usuario", text)

df_clean["text_clean"] = df_clean["text_clean"].apply(anonimizar_menciones_reales)

# Verificación
patron_mencion_real = r"(?<!\w)@\w+(?<!@usuario)"
quedan = df_clean["text_clean"].str.contains(r"(?<!\w)@(?!usuario\b)\w+", regex=True, na=False).sum()
print("Menciones reales sin anonimizar restantes:", quedan)
print("Ejemplos de texto ya anonimizado:")
df_clean[df_clean["text_clean"].str.contains("@usuario", regex=False, na=False)][["text", "text_clean"]].head(10)

Menciones reales sin anonimizar restantes: 0
Ejemplos de texto ya anonimizado:


,text,text_clean
28,"Chino dile a ese pelado que se deje de estarme pidiendo CANJE , como tiene para chupar como pendej@@o🥴🥴🥴 y no tiene para pagar su cuarto","Chino dile a ese pelado que se deje de estarme pidiendo CANJE , como tiene para chupar como pendej@@usuario🥴🥴🥴 y no tiene para pagar su cuarto"
1029,"¶Hace 45 años, nuestro 🇵🇪#PERÚ🇵🇪 desde Morales Bermúdes, pasando por Belaunde Terry, seguido por García Pérez; luego el hampón y genocida Japonés el Yakuza Kenya Inami Inamoto alias ""Alberto Fujimori"" con su siamés y secuas el criminal, delincuente y traídor arequipeño Ex agente de la CIA Judí...","¶Hace 45 años, nuestro 🇵🇪PERÚ🇵🇪 desde Morales Bermúdes, pasando por Belaunde Terry, seguido por García Pérez; luego el hampón y genocida Japonés el Yakuza Kenya Inami Inamoto alias ""Alberto Fujimori"" con su siamés y secuas el criminal, delincuente y traídor arequipeño Ex agente de la CIA JudíoEs..."
3376,"Videaso, que tales monstruos!!! Todos a las @wsl carajo!!! 🙌 respect total","Videaso, que tales monstruos!!! Todos a las @usuario carajo!!! 🙌 respect total"
3456,"Luciano, tienes que pasar por el @Palacio del Sancochado Av. 28 de julio 990 - Cercado. También el Rinconcito Ferreñafano (frente al Congreso). Por el arroz con pato con ceviche de entrada 🙌🏻","Luciano, tienes que pasar por el @usuario del Sancochado Av. 28 de julio 990 - Cercado. También el Rinconcito Ferreñafano (frente al Congreso). Por el arroz con pato con ceviche de entrada 🙌🏻"
3687,❤️Vamos. Michi @micheilleSoifer si se puede estoy segura q vas a seguir en competencia yo te lo aseguro un saludo muy grande desde la. Ciudad de Casma,❤️Vamos. Michi @usuario si se puede estoy segura q vas a seguir en competencia yo te lo aseguro un saludo muy grande desde la. Ciudad de Casma
4260,Quieren hacer Fraude Televisa y La Casa de los Famosos México Están protegiendo a su consentido !! Esto ya es #Viral Fuera Poul) Fueraaaa!! 😤😤😤😤😤😤 Sergio Mayer Breton @ Emilio !!! Que Mexico alce la Voz💢💢💢💢💢💢@Niurka,Quieren hacer Fraude Televisa y La Casa de los Famosos México Están protegiendo a su consentido !! Esto ya es Viral Fuera Poul) Fueraaaa!! 😤😤😤😤😤😤 Sergio Mayer Breton @ Emilio !!! Que Mexico alce la Voz💢💢💢💢💢💢@usuario
4731,"Saludos chicas , sería excelente ganar ese parlante de la gran industria de sonido Marshall debe sonar fabuloso ojalá tenga la suerte con @joaboloña la rubia hermosa...cuídense besos! #MASMUSICACONMARSHALL in one 🌎","Saludos chicas , sería excelente ganar ese parlante de la gran industria de sonido Marshall debe sonar fabuloso ojalá tenga la suerte con @usuario la rubia hermosa...cuídense besos! MASMUSICACONMARSHALL in one 🌎"
4957,"Muy bien, @Gerardo Pe' da gusto cuando lo pones punche sigue así estuvo entretenido tienes un like 👍","Muy bien, @usuario Pe' da gusto cuando lo pones punche sigue así estuvo entretenido tienes un like 👍"
5510,@Un tombo pasó por ahí en una moto deportiva 🤣,@usuario tombo pasó por ahí en una moto deportiva 🤣
5670,"Lo de ese asqueroso de Richard swing debe sancionarse si o si, pero no sean pendejos, en otros lados siempre hacían lo mismo y nadie decía nada...pero, Deplorable lo de #Swing, pero vomitivo que lo condene carla garcia... la hija del ex-Presidente que contrató (de pura 🔥calentura 🔥) a una rec...","Lo de ese asqueroso de Richard swing debe sancionarse si o si, pero no sean pendejos, en otros lados siempre hacían lo mismo y nadie decía nada...pero, Deplorable lo de Swing, pero vomitivo que lo condene carla garcia... la hija del ex-Presidente que contrató (de pura 🔥calentura 🔥) a una reconoc..."


## Verificar textos vacios

In [10]:
# Revisar textos vacíos tras la limpieza
vacios = df_clean[
    df_clean["text_clean"].isna() | (df_clean["text_clean"].astype(str).str.strip() == "")
]
print("Textos vacíos tras limpieza:", len(vacios))

# Revisar textos muy cortos (menos de 3 caracteres) como señal de alerta
muy_cortos = df_clean[df_clean["text_clean"].astype(str).str.len() < 3]
print("Textos con menos de 3 caracteres tras limpieza:", len(muy_cortos))
muy_cortos[["text", "text_clean", "label_name"]]

Textos vacíos tras limpieza: 0
Textos con menos de 3 caracteres tras limpieza: 0


,text,text_clean,label_name


In [11]:
# Verificación final: nulos, duplicados en text_clean, y distribución de clases
print("Nulos por columna:")
print(df_clean.isnull().sum())

print("\nDuplicados en text_clean (pueden aparecer si dos textos distintos quedan iguales tras limpiar):")
duplicados_text_clean = df_clean["text_clean"].duplicated().sum()
print(duplicados_text_clean)

print("\nDistribución de clases en el pool limpio:")
print(df_clean["label_name"].value_counts())
print((df_clean["label_name"].value_counts(normalize=True) * 100).round(2))

print("\nTotal de registros en el pool limpio final:", len(df_clean))

Nulos por columna:
text              0
label             0
label_name        0
split_original    0
text_clean        0
dtype: int64

Duplicados en text_clean (pueden aparecer si dos textos distintos quedan iguales tras limpiar):
0

Distribución de clases en el pool limpio:
label_name
negative    6636
positive    4732
neutral     3176
Name: count, dtype: int64
label_name
negative    45.63
positive    32.54
neutral     21.84
Name: proportion, dtype: float64

Total de registros en el pool limpio final: 14544


In [12]:
# Guardamos el pool limpio final (antes del split 80/10/10)
df_clean_final = df_clean[["text", "text_clean", "label", "label_name"]].copy()

df_clean_final.to_csv("smps_clean_pool.csv", index=False)
df_clean_final.to_parquet("smps_clean_pool.parquet", index=False)

print("Guardado. Total de registros en el pool limpio final:", len(df_clean_final))

Guardado. Total de registros en el pool limpio final: 14544
